#Instructions

1. Suggestions for improvements upon this tool can be found at https://docs.google.com/document/d/12W3JFKD44MdD92Om0t_Q_teH9rECru4LC-4NwclIR4s
1.   Create a shortcut to the [CEM VR Model](https://drive.google.com/drive/folders/187y0SkkZSRGYqBbfmEcxo4atl_ItnQkl?usp=drive_link) folder
  1. Click on "CEM VR Model" in Google Drive
  2. Click on "Organize"
  3. Click on "Add shortcut"
  4. Save it to "My Drive" or another folder
2. Make a copy of this notebook (Double Layer Mask Tool.ipynb)
  1. Click on "Copy to Drive"
3. Adjust input and output files (IMAGE_DIR, OUTPUT_DIR)
  1. Under the block "Manage Input and Output", change the following directories:\
    a. IMAGE_DIR should be the path to your input images\
    b. OUTPUT_DIR should be the desired location for the output\
    c. Adjust the MODEL_PATH and CONFIG_PATH if needed
4. Connect to GPU (T4 GPU, A100 GPU, or V100 GPU)
5. Run all cells
  1. Click on "Runtime"
  2. Click on "Run all"

# Environment Setup


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install torch torchvision

In [ ]:
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
import torch, detectron2
import matplotlib.pyplot as plt
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

#Manage Input and Output


In [17]:
MODEL_PATH = '/content/drive/My Drive/CEM VR Model/model_final.pth'
CONFIG_PATH = '/content/drive/My Drive/config/config.yaml'
IMAGE_DIR = '/content/drive/My Drive/HeadstoneInput/SANC-HeadstonePhotos'
TEST_IMAGE_DIR = '/content/drive/My Drive/testInput'
TEST_OUTPUT_DIR = '/content/drive/My Drive/DebugOutput'
DEMO_IMAGE_DIR = '/content/drive/My Drive/INSTRUCTOR-DEMO/Input'
DEMO_OUTPUT_DIR = '/content/drive/My Drive/INSTRUCTOR-DEMO/Output'
OUTPUT_DIR = '/content/drive/My Drive/testoutput/'

## Set Up Model

In [18]:
import os
import cv2
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from google.colab.patches import cv2_imshow
from natsort import natsorted
import glob
import math

# Set the metadata for your dataset
MetadataCatalog.get("headstones").set(thing_classes=["Headstone", "Standard", "Grass", "Unique"])
metadata = MetadataCatalog.get("headstones")

# Initialize Detectron2 configuration and predictor
cfg = get_cfg()
cfg.merge_from_file(CONFIG_PATH)  # Specify the path to your configuration file
cfg.MODEL.WEIGHTS = MODEL_PATH  # Specify the path to your model weights
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # Set detection threshold
predictor = DefaultPredictor(cfg)

# Ensure the output directory exists
os.makedirs(DEMO_OUTPUT_DIR, exist_ok=True)

# Get list of image paths
image_paths = natsorted(glob.glob(os.path.join(DEMO_IMAGE_DIR, '*.jpg')))  # Adjust file extension if needed

## Pass One

### Detect Bounding Boxes

In [19]:
def pass_one(img):

  #Get Image Shape
  height, width, channels = img.shape

  #Initialize Predictor
  output = predictor(img)

  category_ids = output["instances"].pred_classes.to("cpu").numpy()

  #Create Visualizer
  v = Visualizer(img[:, :, ::-1], metadata=metadata, scale=0.25)
  out = v.draw_instance_predictions(output["instances"].to("cpu"))
  vis_image = out.get_image()[:, :, ::-1]

  #Show Image
  cv2_imshow(vis_image)

  #Cut image to box coordinates
  box = output["instances"].pred_boxes.tensor.cpu().numpy()
  bbox = [int(box[0][0]), int(box[0][1]), int(box[0][2]), int(box[0][3])] #Simple 2D to 1D conversion. Using only the first instance of bounding boxes ensures that we use the box with the highest degree of confidence.

  #SET PADDING HERE
  padding = 100

  #Pad output
  #TODO: do this better please
  if (bbox[0] - padding) >= 0:
    bbox[0] -= padding
  if (bbox[1] - padding) >= 0:
    bbox[1] -= padding
  if (bbox[2] + padding) <= width:
    bbox[2] += padding
  if (bbox[3] + padding) <= height:
    bbox[3] += padding

  #Crop Image
  cropped_img = img[bbox[1]:bbox[3], bbox[0]:bbox[2]]
  return cropped_img, category_ids[0]

## Pass Two

### Detect Masks

In [20]:
def pass_two(cropped_img):

  outputs = predictor(cropped_img)

  # Create a Visualizer object for the current image
  v = Visualizer(cropped_img[:, :, ::-1], metadata=metadata, scale=0.25)

  # Draw the instance predictions (masks and labels) on the image
  out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
  vis_image = out.get_image()[:, :, ::-1]  # Convert image from RGB back to BGR
  out = outputs["instances"].pred_masks.cpu().numpy()
  box = outputs["instances"].pred_boxes.tensor.cpu().numpy()
  cv2_imshow(vis_image)

  save_img = mask_image(out[0], box[0], cropped_img)

  return save_img

### Cut Out Mask

In [21]:
def mask_image(mask, box, original_image):
  mask_h = int(math.ceil(box[3] - box[1]))
  mask_w = int(math.ceil(box[2] - box[0]))

  #Detect & Fix Holes in Masks
  mask = fill_mask_holes(mask)

  #Get Mask pixel by pixel
  temp_mask = numpy.zeros((mask_h, mask_w))
  for x_idx in range(int(box[1]), int(box[3])):
    for y_idx in range(int(box[0]), int(box[2])):
      temp_mask[x_idx - int(box[1])][y_idx - int(box[0])] = mask[x_idx][y_idx]

  temp_mask_ints = temp_mask.astype(int)

  #overlay mask with original image
  temp_mask_fill = numpy.zeros((mask_h, mask_w, 4))
  for x_idx, h_bw in enumerate(temp_mask_ints):
    for y_idx, w_bw in enumerate(h_bw):
      if(w_bw == 0):
        temp_mask_fill[x_idx][y_idx] = [0, 0, 0, 0]
      else:
        orig_w = int(math.ceil(y_idx + box[0]))
        orig_h = int(math.ceil(x_idx + box[1]))
        temp_mask_fill[x_idx][y_idx] = numpy.append(original_image[orig_h - 1][orig_w - 1], 1)
  cv2_imshow(temp_mask_fill[:,:,0:3])
  return temp_mask_fill

In [22]:
from scipy.ndimage import binary_fill_holes

#Fills Holes in mask
def fill_mask_holes(mask):

  mask_np = mask
  filled_mask = binary_fill_holes(mask_np).astype(numpy.uint8)

  return filled_mask

## Run Model

### Get Starting Point From User

In [23]:
def get_skip_input():

  #Get User Target and set flag
  target_img = input("Specify image name to begin processing from (Type 0 to process all images in source folder): ")

  do_skip = True #Used to stop skipping loop once target has been reached

  if target_img == "0":
    do_skip = False

  return target_img, do_skip

### Start Model



In [ ]:
# Iterate over sorted image paths
import numpy

target_img, do_skip = get_skip_input()

for image_path in image_paths:
    img = cv2.imread(image_path)

    #Loop until target image is found, then go as normal
    if do_skip and (target_img in image_path) == False:
      continue;
    do_skip = False

    #First Pass
    cropped_img, stone_type = pass_one(img)

    #Target Standard Headstones
    if stone_type == 1:

      save_img = pass_two(cropped_img)

      # Save the visualized image
      save_path = os.path.join(DEMO_OUTPUT_DIR, os.path.basename(image_path))
      cv2.imwrite(save_path, save_img)

print(f"Processed images. Visualized outputs saved to {OUTPUT_DIR}")